In [13]:
# Q3: Vision Transformer and CLIP on a Classification Task

In [14]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

In [15]:
# transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Load dataset
train_dataset = datasets.ImageFolder(
    root="seg_train",
    transform=transform
)

test_dataset = datasets.ImageFolder(
    root="seg_test",
    transform=transform
)

# dataloaders
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

# load vit
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)

def to_pil_batch(tensor_batch):
    return [transforms.ToPILImage()(img.cpu()) for img in tensor_batch]

# freeze 
for param in model.parameters():
    param.requires_grad = False

In [16]:
# 2. Fine-tune a pretrained Vision Transformer from torchvision on the Intel dataset by
#freezing all pretrained layers and replacing the classification head with a new linear
#layer for 6 output classes, training for at least 5 epochs using CrossEntropyLoss and
#the Adam optimizer.

# classifer
num_features = model.heads.head.in_features
model.heads.head = nn.Linear(num_features, 6)
model = model.to(device)

# loss and optiizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.heads.parameters(), lr=1e-3)

# train
epochs = 5

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

# evaluate vit accurary
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        preds = outputs.argmax(dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

vit_acc = correct / total
print("ViT Accuracy:", vit_acc)

Epoch 1, Loss: 0.0188
Epoch 2, Loss: 0.0006
Epoch 3, Loss: 0.0002
Epoch 4, Loss: 0.0001
Epoch 5, Loss: 0.0000
ViT Accuracy: 1.0


In [17]:

#3. Implement zero-shot classification using OpenAI’s CLIP model on the same dataset by
#encoding each of the 6 class names as text prompts (e.g., “a photo of a glacier”) and
#comparing them against image embeddings to predict the class without any fine-tuning.

from transformers import CLIPProcessor, CLIPModel
from PIL import Image

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval().to(device)

# define intel labels
labels = [
    "a photo of a building",
    "a photo of a forest",
    "a photo of a glacier",
    "a photo of a mountain",
    "a photo of a sea",
    "a photo of a street"
]

# clip prediction function
def clip_predict_batch(images):
    # ensure correct format
    if isinstance(images, list) == False:
        images = [images]

    inputs = clip_processor(
        text=labels,
        images=images,
        return_tensors="pt",
        padding=True
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = clip_model(**inputs)
        probs = outputs.logits_per_image.softmax(dim=1)

    return probs.argmax(dim=1).detach().cpu()

# evaluate clip accurary
correct = 0
total = 0

with torch.no_grad():
    for images, labels_true in test_loader:

        # convert tensor images → PIL images
        images_pil = to_pil_batch(images)

        # CLIP prediction
        preds = clip_predict_batch(images_pil)

        # accuracy counting
        correct += (preds == labels_true).sum().item()
        total += labels_true.size(0)

clip_acc = correct / total
print("CLIP Accuracy:", clip_acc)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIP Accuracy: 0.03933333333333333


In [18]:

#4. Evaluate both models on the test set by computing accuracy, and visualize 5 test
#images with their true labels, ViT predicted labels, and CLIP predicted labels displayed
#together in a single figure.

class_names = train_dataset.classes

import matplotlib.pyplot as plt

images, labels = next(iter(test_loader))

images = images.to(device)

vit_preds = model(images).argmax(dim=1)

plt.figure(figsize=(12,6))

for i in range(5):
    img = images[i].cpu().permute(1,2,0)

    # ViT prediction
    vit_label = class_names[vit_preds[i]]

    # CLIP prediction
    img_pil = transforms.ToPILImage()(images[i].cpu())
    
    clip_pred = clip_predict_batch([img_pil])[0].item()
    clip_label = class_names[int(clip_pred)]

    true_label = class_names[labels[i]]

    plt.subplot(2,5,i+1)
    plt.imshow(img)
    plt.title(f"True: {true_label}")
    plt.axis("off")

    plt.subplot(2,5,i+6)
    plt.imshow(img)
    plt.title(f"ViT: {vit_label}\nCLIP: {clip_label}")
    plt.axis("off")

plt.tight_layout()
plt.show()

ValueError: text input must be of type `str` (single example), `list[str]` (batch or single pretokenized example) or `list[list[str]]` (batch of pretokenized examples) or `list[tuple[list[str], list[str]]]` (batch of pretokenized sequence pairs).

<Figure size 1200x600 with 0 Axes>

In [ ]:
#5. Analyze the results by discussing how ViT’s fine-tuned classification compares to
#CLIP’s zero-shot performance, what the accuracy difference reveals about the value
#of task-specific training versus large-scale contrastive pretraining, and reflect on which
#approach would be more practical in a real-world scenario with limited labeled data.

In [ ]:
# ViT uses fine-tuned training on the labeled dataset, so it learns task-specific features and usually achieves higher accuracy because it is directly optimized for the target classes.  
# CLIP uses zero-shot classification with large-scale contrastive pretraining, meaning it does not learn from the task dataset but instead matches images to text descriptions.  
# As a result, ViT typically performs better in accuracy when enough labeled data is available, while CLIP is more flexible but less precise for specific tasks.  
# The accuracy difference shows that task-specific training is more effective when labeled data exists, while large-scale pretraining provides strong generalization without training.  
# In real-world scenarios with limited labeled data, CLIP is more practical because it works without retraining, but ViT becomes better when enough labeled data is available for fine-tuning.  